In [ ]:
# 🧠 Zentro – Credit Default Risk Prediction
This project uses Fannie Mae historical loan data to build a machine learning model that predicts the probability of loan default. # type: ignore


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import xgboost as xgb
import shap


In [6]:
import pandas as pd

df = pd.read_csv("2024Q4.csv", sep="|", header=None)
df.head()


/var/folders/m0/whtgxt194wncw5j97grk0vl80000gn/T/ipykernel_5984/1252978787.py:3: DtypeWarning: Columns (101,104) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("2024Q4.csv", sep="|", header=None)


,0,1,2,3,4,5,6,7,8,9,...,100,101,102,103,104,105,106,107,108,109
0,NaN,138923847,102024,R,Other,Nationstar Mortgage LLC,NaN,6.990,6.990,618000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
1,NaN,138923847,112024,R,Other,Nationstar Mortgage LLC,NaN,6.990,6.990,618000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
2,NaN,138923847,122024,R,Other,Nationstar Mortgage LLC,NaN,6.990,6.990,618000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
3,NaN,138923848,102024,B,Other,Other,NaN,5.625,5.625,419000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
4,NaN,138923848,112024,B,Other,Other,NaN,5.625,5.625,419000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN


In [8]:
import pandas as pd

glossary = pd.read_excel("crt-file-layout-and-glossary.xlsx")
print(glossary.columns)
print(glossary.head())


Index(['Field Position', 'Field Name', 'Description', 'Enumerations',
       'Date Bound Notes', 'Respective Disclosure Notes', 'CAS', 'CIRT',
       'Single-Family (SF) Loan Performance', 'Type', 'Max Length'],
      dtype='object')
   Field Position                Field Name  \
0               1         Reference Pool ID   
1               2           Loan Identifier   
2               3  Monthly Reporting Period   
3               4                   Channel   
4               5               Seller Name   

                                         Description  \
0        A unique identifier for the reference pool.   
1         A unique identifier for the mortgage loan.   
2  The month and year that pertains to the servic...   
3  The origination channel used by the party that...   
4  The name of the entity that delivered the mort...   

                                Enumerations  \
0                                        NaN   
1                                        NaN   
2 

In [9]:
df = pd.read_csv("2024Q4.csv", sep="|", header=None)
column_names = glossary['Field Name'].tolist()

# Если количество совпадает (110), переименовываем
df.columns = column_names

df.head()


/var/folders/m0/whtgxt194wncw5j97grk0vl80000gn/T/ipykernel_5984/2054343962.py:1: DtypeWarning: Columns (101,104) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("2024Q4.csv", sep="|", header=None)


,Reference Pool ID,Loan Identifier,Monthly Reporting Period,Channel,Seller Name,Servicer Name,Master Servicer,Original Interest Rate,Current Interest Rate,Original UPB,...,ARM Plan Number,Borrower Assistance Plan,High Loan to Value (HLTV) Refinance Option Indicator,Deal Name,Repurchase Make Whole Proceeds Flag,Alternative Delinquency Resolution,Alternative Delinquency Resolution Count,Total Deferral Amount,Payment Deferral Modification Event Indicator,Interest Bearing UPB
0,NaN,138923847,102024,R,Other,Nationstar Mortgage LLC,NaN,6.990,6.990,618000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
1,NaN,138923847,112024,R,Other,Nationstar Mortgage LLC,NaN,6.990,6.990,618000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
2,NaN,138923847,122024,R,Other,Nationstar Mortgage LLC,NaN,6.990,6.990,618000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
3,NaN,138923848,102024,B,Other,Other,NaN,5.625,5.625,419000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
4,NaN,138923848,112024,B,Other,Other,NaN,5.625,5.625,419000.0,...,NaN,7,N,NaN,NaN,7,NaN,NaN,7,NaN
